In [ ]:
# TODO: add multi layer support

CONFIG = {
    "block_size" : 64,
    "n_embed" : 128,
    "batch_size" : 64,
    "hidden_size" : 256,  # Increased from 10 to 256 for better capacity
    "n_epochs" : 50,      # Increased from 10 to 50    "batch_size" : 64,
    "learning_rate" : 3e-3,  # Increased from 1e-4 to 3e-3
}

In [2]:
import torch
DEVICE = torch.device("mps")

In [3]:
# load shakespear
with open("data/lotr.txt", "r", encoding="utf-8") as f: DATASET_TEXT = f.read()
DATASET_TEXT[:10000]

'In this reprint several minor inaccuracies, most of them noted by readers, have been corrected. For example, the text on pages 32 and 62 now corresponds exactly with the runes on Thror\'s Map. More important is the matter of Chapter Five. There the true story of the ending of the Riddle Game, as it was eventually revealed (under pressure) by Bilbo to Gandalf, is now given according to the Red Book, in place of the version Bilbo first gave to his friends, and actually set down in his diary. This departure from truth on the part of a most honest hobbit was a portent of great significance. It does not, however, concern the present story, and those who in this edition make their first acquaintance with hobbit-lore need not troupe about it. Its explanation lies in the history of the Ring, as it was set out in the chronicles of the Red Book of Westmarch, and is now told in The Lord of the Rings.\n\nA final note may be added, on a point raised by several students of the lore of the period. O

In [4]:
# create tokenizer
VOCAB = sorted(list(set(DATASET_TEXT)))
ctoi = dict([(c, i) for i, c in enumerate(VOCAB)])
itoc = dict([(i, c) for i, c in enumerate(VOCAB)])
encode = lambda str: [ctoi[c] for c in str]
decode = lambda tokens: "".join([itoc[i] for i in tokens])
len(VOCAB), decode(encode("hello world"))

(108, 'hello world')

In [5]:
# tokenize shakespear
DATASET_TOKENS = encode(DATASET_TEXT)
len(DATASET_TOKENS), DATASET_TOKENS[:10]

(3301752, [36, 72, 1, 78, 66, 67, 77, 1, 76, 63])

In [6]:
import torch
from torch.utils.data import Dataset

class LOTRDataset(Dataset):
    def __init__(self, tokens, block_size=10):
        super().__init__()
        tokens = torch.tensor(tokens)
        chunk_size = block_size + 1
        n_chunks = len(tokens) // chunk_size
        chunks = tokens[:chunk_size * n_chunks]
        chunks = chunks.view(n_chunks, chunk_size)
        self.chunks = chunks
    
    def __getitem__(self, idx):
        tokens = self.chunks[idx]
        x = tokens[:-1]
        y = tokens[1:]
        return x, y
        
    def __len__(self):
        return len(self.chunks)

# Fixed: Pass block_size from CONFIG instead of using default 10
DATASET = LOTRDataset(DATASET_TOKENS, block_size=CONFIG["block_size"])
for i in range(10):
    x, y = DATASET[i]
    print(decode(x.tolist()))
    print(decode(y.tolist()))
    print("---")

In this reprint several minor inaccuracies, most of them noted b
n this reprint several minor inaccuracies, most of them noted by
---
 readers, have been corrected. For example, the text on pages 32
readers, have been corrected. For example, the text on pages 32 
---
and 62 now corresponds exactly with the runes on Thror's Map. Mo
nd 62 now corresponds exactly with the runes on Thror's Map. Mor
---
e important is the matter of Chapter Five. There the true story 
 important is the matter of Chapter Five. There the true story o
---
f the ending of the Riddle Game, as it was eventually revealed (
 the ending of the Riddle Game, as it was eventually revealed (u
---
nder pressure) by Bilbo to Gandalf, is now given according to th
der pressure) by Bilbo to Gandalf, is now given according to the
---
 Red Book, in place of the version Bilbo first gave to his frien
Red Book, in place of the version Bilbo first gave to his friend
---
s, and actually set down in his diary. This departure from tru

In [7]:
from torch.utils.data import DataLoader

dataloader = DataLoader(
    DATASET,
    batch_size=CONFIG["batch_size"],
    shuffle=True
)
print(len(DATASET))
for x,y in dataloader:
    for i in range(x.shape[0]):
        print(decode(x[i].tolist()))
        print(decode(y[i].tolist()))
        print("----")
        break

50796
feeling rather pleased with the cleverness of his conversation w
eeling rather pleased with the cleverness of his conversation wi
----
croaching on its borders and supporting its enemies.

2948-3019 
roaching on its borders and supporting its enemies.

2948-3019 1
----
an. For that meant 'Dwarf-delving' and yet was already word of a
n. For that meant 'Dwarf-delving' and yet was already word of an
----
e river, gurgling away in the black shadows under its deep banks
 river, gurgling away in the black shadows under its deep banks.
----
ndybucks, and Grubbs, and Chubbs, and Burrowses, and Hornblowers
dybucks, and Grubbs, and Chubbs, and Burrowses, and Hornblowers,
----
g," said Merry. "I should not sing any more at present. Wait til
," said Merry. "I should not sing any more at present. Wait till
----
h the Mouth of Sauron!" he cried. "Surety you crave! Sauron give
 the Mouth of Sauron!" he cried. "Surety you crave! Sauron gives
----
esent weak obscured vowels) is here employed in t

In [8]:
import torch.nn as nn

class RNNCell(nn.Module):
    def __init__(self, hidden_size=CONFIG["hidden_size"], n_embed=CONFIG["n_embed"]):
        super().__init__()
        self.n_embed = n_embed
        self.hidden_size = hidden_size

        self.Wx = nn.Linear(n_embed, hidden_size, bias=True)
        self.Wh = nn.Linear(hidden_size, hidden_size, bias=True)
    
    def forward(self, x):
        B, T, C = x.shape

        outputs = []
        n_t = x.shape[1]
        ht = torch.zeros((B, self.hidden_size)).to(DEVICE)

        x_transformed = self.Wx(x)
        
        hidden_states = []
        for t in range(n_t):
            xt_hat = x_transformed[:, t, :]
            ht_hat = self.Wh(ht)
            ht = torch.tanh(xt_hat + ht_hat)
            hidden_states.append(ht)
            outputs.append(ht)
        outputs = torch.stack(outputs, dim=1) # TODO: why the dim=1?
        hidden_states = torch.stack(hidden_states, dim=1)

        return outputs, hidden_states
        
class RNN(nn.Module):
    def __init__(self, vocab_size, hidden_size=CONFIG["hidden_size"], n_embed=CONFIG["n_embed"]):
        super().__init__()
        self.n_embed = n_embed
        self.hidden_size = hidden_size
        self.embeddings = nn.Embedding(vocab_size, n_embed)

        # Replace Wx + Wh + loop with PyTorch's RNN
        #self.rnn = nn.RNN(
        #    input_size=n_embed,      # Takes embedding dimension as input
        #    hidden_size=hidden_size,  # Outputs hidden_size dimension
        #    num_layers=1,             # Single layer
        #    batch_first=True,         # Input/output: (batch, seq, feature)
        #    nonlinearity='tanh'       # Same activation as your implementation
        #)
        self.rnn = RNNCell(
            n_embed=n_embed,
            hidden_size=hidden_size,
        )

        self.ffn = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        x_emb = self.embeddings(x)
        outputs, _ = self.rnn(x_emb)
        logits = self.ffn(outputs)
        return logits
        

VOCAB_SIZE = len(ctoi)
x, y = next(iter(dataloader))
x = x.to(DEVICE)
model = RNN(VOCAB_SIZE).to(DEVICE)
model(x)

tensor([[[ 0.3137,  0.0195,  0.0827,  ...,  0.1137,  0.2343,  0.6255],
         [-0.0196,  0.0279,  0.3030,  ...,  0.4649,  0.3372,  0.2336],
         [ 0.0343,  0.4116,  0.0576,  ...,  0.0345,  0.4422,  0.3415],
         ...,
         [ 0.4383,  0.5726, -0.1709,  ...,  0.1143, -0.3731,  0.2549],
         [ 0.4716,  0.0116, -0.0552,  ..., -0.2081,  0.2633,  0.5558],
         [-0.2224,  0.0264,  0.0867,  ...,  0.2735,  0.1432,  0.4827]],

        [[ 0.0870, -0.0937,  0.1421,  ...,  0.0497, -0.0773, -0.5564],
         [ 0.0757,  0.3012, -0.1738,  ..., -0.5250,  0.1662,  0.2656],
         [-0.0617, -0.3253,  0.3818,  ..., -0.2121, -0.1127,  0.1656],
         ...,
         [ 0.2481,  0.0097,  0.0207,  ...,  0.1970, -0.3705, -0.6725],
         [ 0.0335,  0.2913, -0.0441,  ..., -0.1757, -0.0097,  0.3979],
         [ 0.1277,  0.5151,  0.7466,  ..., -0.1607,  0.2121,  0.1177]],

        [[ 0.3784,  0.6979,  0.4826,  ..., -0.1318,  0.1526, -0.1444],
         [ 0.2479, -0.0609,  0.0384,  ...,  0

In [9]:
total_params = sum(p.numel() for p in model.parameters())
print(total_params)

140396


In [10]:
total = 0
for name, param in model.named_parameters():
    total += param.numel()
    print(name, param.numel())
print(total)

embeddings.weight 13824
rnn.Wx.weight 32768
rnn.Wx.bias 256
rnn.Wh.weight 65536
rnn.Wh.bias 256
ffn.weight 27648
ffn.bias 108
140396


In [11]:
import torch.nn.functional as F

# TODO: must keep context window cropped to block size
# TODO: max len must take promtp into account
# TODO: this can be sped up
def generate(model, prompt, block_size=CONFIG["block_size"], max_len=None):
    print(prompt, end="")
    tokens = encode(prompt)

    if max_len is None: max_len = block_size - len(tokens)
    tokens = torch.tensor(tokens, device=DEVICE).unsqueeze(0)
    for _ in range(max_len):
        logits = model(tokens).squeeze(0)
        probs = F.softmax(logits, dim=-1)
        idxs = torch.multinomial(probs, num_samples=1)
        token = idxs[-1, :]
        token = token.item()
        print(decode([token]), end="")
        token_t = torch.tensor(token, device=DEVICE).unsqueeze(0).unsqueeze(0)
        tokens = torch.cat([tokens, token_t], dim=-1)
generate(model, "Frodo picked up")

Frodo picked upXHA
SqEä|g~ysûûYá-LXMIû&^vQ16f0r1z
PPWSq7rô3s22o[

In [12]:
model = RNN(VOCAB_SIZE)
model = model.to(DEVICE)
model

RNN(
  (embeddings): Embedding(108, 128)
  (rnn): RNNCell(
    (Wx): Linear(in_features=128, out_features=256, bias=True)
    (Wh): Linear(in_features=256, out_features=256, bias=True)
  )
  (ffn): Linear(in_features=256, out_features=108, bias=True)
)

In [13]:
from tqdm import tqdm
from torch.nn.utils import clip_grad_norm_

lr = CONFIG["learning_rate"]
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
losses = []
gradnorms = []
leave = False
for epoch in range(CONFIG["n_epochs"]):
    bar = tqdm(dataloader) # TODO: reuse bar
    if leave: break
    for x, y in bar:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        #logits = F.softmax(logits, dim=-1)
        #y_pred = torch.multinomial(logits, dim=-1, num_samples=1)
        logits = logits.permute(0, 2, 1) # -> [N, C, L]      (64, 108, 10)
        loss = F.cross_entropy(logits, y)
        optimizer.zero_grad()
        loss.backward()

        total_norm = clip_grad_norm_(model.parameters(), max_norm=float('inf'))
        gradnorms.append(total_norm.item())

        optimizer.step()
        loss_i = loss.item()
        bar.set_postfix(dict(epoch=epoch, loss=loss_i))
        losses.append(loss_i)
        #print(loss)


 11%|█         | 88/794 [00:02<00:16, 43.44it/s, epoch=5, loss=1.33]


KeyboardInterrupt: 

In [ ]:
from matplotlib import pyplot as plt

fig, ax = plt.subplots(1, 2)

ax[0].plot(losses)
ax[1].plot(gradnorms)


In [ ]:
generate(model, "frodo", max_len=1024)